## Task importer tutorial

This notebook mirrors the flow of `importer_example.ipynb`, but for **tasks**: definitions, tasks, subtasks, and links to existing **image instances**.

The task graph is **independent** from the image hierarchy. Rows are `ImportTaskRow` (not `ImportRow`). Task runs use `plan_import` with `entity_specs=TASK_ENTITY_SPECS`. Image runs use `plan_image_import` (or `build_image_import_rows` then `plan_import`).

- Image rows: `plan_image_import(session, rows, defaults=..., infer_storage_format=...)` (wraps `prepare_rows` + `plan_import`).

Entity chain (simplified):

`TaskDefinition` → `Task` (optional `Contact` / `Creator`) → `SubTask` (optional `Creator`) → `SubTaskImageLink` (`ImageInstanceID` is a plain FK — no `ImageInstance` entity in this graph).

Typical workflow:

1. Ensure target `ImageInstance` rows exist (e.g. import images with `ImportRow` + default `ENTITY_SPECS`).
2. Build flat `ImportTaskRow` lists, or use `expand_task_import_rows` when many rows share the same task fields.
3. `plan_import(session, rows, entity_specs=TASK_ENTITY_SPECS)` → `ImportRun`.
4. Review (`run.display_summary()`), then `run.apply()` and `session.commit()`.

Notes:

- Anonymous subtasks use a batch-local integer `subtask_anonymous_identity` to group link rows under the same new `SubTask`.
- Nothing is persisted until `apply()` + `commit()`.

In [ ]:
from datetime import date

from sqlalchemy import func, select
from sqlalchemy.orm import Session

from eyened_orm import (
    ImageInstance,
    SubTask,
    SubTaskImageLink,
    Task,
    TaskDefinition,
)
from eyened_orm.importer import (
    ImportRow,
    ImportTaskRow,
    expand_task_import_rows,
    plan_image_import,
    plan_import,
)
from eyened_orm.importer.importer_mappings_tasks import TASK_ENTITY_SPECS
from eyened_orm.utils.sqlite_testdb import create_sqlite_memory_sessionmaker

SessionLocal = create_sqlite_memory_sessionmaker(expire_on_commit=False)
session: Session = SessionLocal()
print("Ready: in-memory SQLite session")


def count(session: Session, model) -> int:
    return session.scalar(select(func.count()).select_from(model))


def print_task_counts(session: Session) -> None:
    for model in (TaskDefinition, Task, SubTask, SubTaskImageLink, ImageInstance):
        print(f"  {model.__name__:<20} : {count(session, model)}")

Ready: in-memory SQLite session


## 1. Create a few images (get real `ImageInstanceID` values)

The task importer only stores **foreign keys** to images. Import a small image batch first, then read IDs from the database (or from your application).

In [2]:
img_defaults = {
    "project_external": "Y",
    "manufacturer": "tutorial",
    "manufacturer_model_name": "tutorial-model",
    "device_description": "tutorial-device",
    "dataset_identifier": "",
    "storage_backend_kind": "local",
}

d = date(2026, 5, 1)
img_rows = [
    ImportRow(
        project_name="task-tutorial-project",
        patient_identifier="task-tutorial-patient",
        study_date=d,
        series_instance_uid="task-tutorial-series",
        storage_backend_key="task-tutorial-sb",
        object_key=f"task-tutorial-{i}.png",
        modality="ColorFundus",
        laterality="L",
    )
    for i in range(4)
]

run_img = plan_image_import(session, img_rows, defaults=img_defaults)
run_img.display_summary()
run_img.apply()
session.commit()

image_ids = list(
    session.scalars(
        select(ImageInstance.ImageInstanceID).order_by(ImageInstance.ImageInstanceID)
    ).all()
)
print("ImageInstanceIDs:", image_ids)
print_task_counts(session)

Entity,Update,Create,Total
ImageInstance,,4,4
ImageStorage,,4,4
StorageBackend,,1,1
DeviceModel,,1,1
Project,,1,1
DeviceInstance,,1,1
Patient,,1,1
Study,,1,1
Series,,1,1


ImageInstanceIDs: [1, 2, 3, 4]
  TaskDefinition       : 0
  Task                 : 0
  SubTask              : 0
  SubTaskImageLink     : 0
  ImageInstance        : 4


## 2. Flat `ImportTaskRow` (one row = one `SubTaskImageLink`)

Repeat shared fields (`task_definition_name`, `task_name`, `creator_name`, …) on every row. Use the same `subtask_anonymous_identity` on rows that belong to the same logical subtask; increment `subtask_image_index` per image within that subtask.

In [3]:
i0, i1, i2, i3 = image_ids[0], image_ids[1], image_ids[2], image_ids[3]

flat_task_rows = [
    ImportTaskRow(
        task_definition_name="tutorial-taskdef",
        task_name="tutorial-task",
        creator_name="tutorial-creator",
        subtask_anonymous_identity=1,
        image_instance_id=i0,
        subtask_image_index=0,
    ),
    ImportTaskRow(
        task_definition_name="tutorial-taskdef",
        task_name="tutorial-task",
        creator_name="tutorial-creator",
        subtask_anonymous_identity=1,
        image_instance_id=i1,
        subtask_image_index=1,
    ),
    ImportTaskRow(
        task_definition_name="tutorial-taskdef",
        task_name="tutorial-task",
        creator_name="tutorial-creator",
        subtask_anonymous_identity=2,
        image_instance_id=i2,
        subtask_image_index=0,
    ),
]

run_flat = plan_import(
    session,
    flat_task_rows,
    entity_specs=TASK_ENTITY_SPECS,
)
run_flat.display_summary()
run_flat.apply()
session.commit()

print("After flat task import:")
print_task_counts(session)

Entity,Update,Create,Total
SubTaskImageLink,,3,3
SubTask,,2,2
Creator,,1,1
TaskDefinition,,1,1
Task,,1,1


After flat task import:
  TaskDefinition       : 1
  Task                 : 1
  SubTask              : 2
  SubTaskImageLink     : 3
  ImageInstance        : 4


## 3. Grouped layout → `expand_task_import_rows`

When one task shares many images across a few subtasks, use **one template row** plus a list of image-id **lists**. Each list is one anonymous subtask; `subtask_anonymous_identity` is set automatically to `1`, `2`, … in order.

Example (same task, two subtasks, overlapping images):

- First group: `[a, b, c]` → anonymous identity `1`
- Second group: `[a, d]` → anonymous identity `2`

That expands to five flat `ImportTaskRow` values with correct `subtask_image_index` within each group.

In [4]:
template = ImportTaskRow(
    task_definition_name="tutorial-taskdef-2",
    task_name="tutorial-task-2",
    creator_name="tutorial-creator",
    contact_name="Dr. Example",
    contact_email="example@hospital.org",
    contact_institute="Example Institute",
)

image_groups = [[i0, i1, i2], [i0, i3]]

expanded = expand_task_import_rows(template, image_groups)
print(f"Expanded {len(image_groups)} groups → {len(expanded)} flat rows")
for r in expanded:
    print(
        "  anon=", r.subtask_anonymous_identity,
        "img=", r.image_instance_id,
        "idx=", r.subtask_image_index,
    )

run_grp = plan_import(
    session,
    expanded,
    entity_specs=TASK_ENTITY_SPECS,
)
run_grp.display_summary()
run_grp.apply()
session.commit()

print("After grouped (second task):")
print_task_counts(session)

Expanded 2 groups → 5 flat rows
  anon= 1 img= 1 idx= 0
  anon= 1 img= 2 idx= 1
  anon= 1 img= 3 idx= 2
  anon= 2 img= 1 idx= 0
  anon= 2 img= 4 idx= 1


Entity,Update,Create,Total
SubTaskImageLink,,5,5
SubTask,,2,2
Contact,,1,1
TaskDefinition,,1,1
Task,,1,1


After grouped (second task):
  TaskDefinition       : 2
  Task                 : 2
  SubTask              : 4
  SubTaskImageLink     : 8
  ImageInstance        : 4


## 4. Optional: inspect links

Each `SubTaskImageLink` ties a `SubTask` to an `ImageInstance`.

In [5]:
links = session.scalars(select(SubTaskImageLink)).all()
print(f"Total SubTaskImageLink rows: {len(links)}")
for link in links[:8]:
    print(
        "  SubTaskID=", link.SubTaskID,
        "ImageInstanceID=", link.ImageInstanceID,
        "ImageIndex=", link.ImageIndex,
    )
if len(links) > 8:
    print("  ...")

Total SubTaskImageLink rows: 8
  SubTaskID= 1 ImageInstanceID= 1 ImageIndex= 0
  SubTaskID= 1 ImageInstanceID= 2 ImageIndex= 1
  SubTaskID= 2 ImageInstanceID= 3 ImageIndex= 0
  SubTaskID= 3 ImageInstanceID= 1 ImageIndex= 0
  SubTaskID= 3 ImageInstanceID= 2 ImageIndex= 1
  SubTaskID= 3 ImageInstanceID= 3 ImageIndex= 2
  SubTaskID= 4 ImageInstanceID= 1 ImageIndex= 0
  SubTaskID= 4 ImageInstanceID= 4 ImageIndex= 1


## See also

- `importer_example.ipynb` — image / `ImportRow` pipeline, CSV, undo patterns.
- `eyened_orm/importer/tests/test_task_import_independent.py` and `test_task_import_expand.py` — minimal automated examples.
- `ImportRun` (`display_summary`, `apply`, `write_json`, `undo`) works the same way for task runs.